In [ ]:
import os

os.environ["KAGGLE_API_KEY"] = "YOUR_API_KEY"

In [2]:
import os, shutil, subprocess

SLUG = "mostafaabla/garbage-classification"
RAW = "/content/kaggle_raw"
DST = "/content/data"
MERGE = {"paper": "paper", "cardboard": "paper",
         "plastic": "plastic", "biological": "organic"}

subprocess.run(["pip", "install", "-q", "kaggle"], check=True)
shutil.rmtree(RAW, ignore_errors=True)
subprocess.run(["kaggle", "datasets", "download", "-d", SLUG,
                "-p", RAW, "--unzip"], check=True)

found = {}
for dirpath, dirnames, _ in os.walk(RAW):
    if "__MACOSX" in dirpath:
        continue
    for name in dirnames:
        if name in MERGE and name not in found:
            found[name] = os.path.join(dirpath, name)
print("found:", sorted(found))

shutil.rmtree(DST, ignore_errors=True)
counts = {}
for src, dst in MERGE.items():
    if src not in found:
        print("MISSING:", src)
        continue
    os.makedirs(os.path.join(DST, dst), exist_ok=True)
    for f in os.listdir(found[src]):
        if f.startswith("."):
            continue
        shutil.copy(os.path.join(found[src], f),
                    os.path.join(DST, dst, f"{src}_{f}"))
        counts[dst] = counts.get(dst, 0) + 1
print("images per class:", counts)

found: ['biological', 'cardboard', 'paper', 'plastic']
images per class: {'paper': 1941, 'plastic': 865, 'organic': 985}


In [3]:
import json, os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

DATA_DIR = "/content/data"
OUT_DIR = "/content/out"
IMG = 96
BATCH = 32
SEED = 1
VAL_FRAC = 0.2
EPOCHS_HEAD = 8
EPOCHS_FINE = 15
AUTO = tf.data.AUTOTUNE

os.makedirs(OUT_DIR, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)

# ---------- data ----------
train_raw = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=VAL_FRAC, subset="training", seed=SEED,
    image_size=(IMG, IMG), batch_size=None, label_mode="int")
val_raw = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=VAL_FRAC, subset="validation", seed=SEED,
    image_size=(IMG, IMG), batch_size=None, label_mode="int")
class_names = train_raw.class_names
NUM_CLASSES = len(class_names)

# class weights from the training split
train_labels = [int(l) for _, l in train_raw]
counts = np.bincount(train_labels, minlength=NUM_CLASSES)
class_weight = {i: len(train_labels) / (NUM_CLASSES * c) for i, c in enumerate(counts)}
print("classes:", class_names)
print("train counts:", dict(zip(class_names, counts.tolist())))
print("val images:", int(val_raw.cardinality()))

def esp32_degrade(img, label):
    """Make a clean TrashNet/Kaggle photo look like an OV2640 frame at 96x96."""
    img = tf.cast(img, tf.float32) / 255.0

    # soft optics: downscale then upscale by a random factor
    s = tf.random.uniform([], 48, IMG + 1, dtype=tf.int32)
    img = tf.image.resize(img, (s, s), method="area")
    img = tf.image.resize(img, (IMG, IMG), method="bilinear")

    # white balance and exposure variation
    img *= tf.random.uniform([1, 1, 3], 0.85, 1.15)
    img = tf.image.random_saturation(img, 0.6, 1.4)
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.7, 1.3)

    # sensor noise
    img += tf.random.normal(tf.shape(img), stddev=tf.random.uniform([], 0.0, 0.05))
    img = tf.clip_by_value(img, 0.0, 1.0)

    # RGB565 quantisation, the color depth the ESP32 raw path produces
    r = tf.round(img[..., 0] * 31.0) / 31.0
    g = tf.round(img[..., 1] * 63.0) / 63.0
    b = tf.round(img[..., 2] * 31.0) / 31.0
    img = tf.stack([r, g, b], axis=-1)

    # JPEG artifacts (in case any frame passes through JPEG)
    img8 = tf.image.convert_image_dtype(img, tf.uint8)
    img8 = tf.image.random_jpeg_quality(img8, 30, 90)
    img = tf.image.convert_image_dtype(img8, tf.float32)

    return img * 255.0, label

def geom(img, label):
    # camera is fixed, so only horizontal flip (no vertical flip, no rotation)
    return tf.image.random_flip_left_right(img), label

train = (train_raw.map(esp32_degrade, AUTO).map(geom, AUTO)
         .shuffle(512, seed=SEED).batch(BATCH).prefetch(AUTO))
val_clean = val_raw.batch(BATCH).prefetch(AUTO)
val_deg = val_raw.map(esp32_degrade, AUTO).batch(BATCH).prefetch(AUTO)

# ---------- model ----------
# No augmentation layers inside the model, so int8 conversion stays simple.
base = tf.keras.applications.MobileNetV2((IMG, IMG, 3), alpha=0.35,
                                         include_top=False, weights="imagenet")
base.trainable = False

inputs = layers.Input((IMG, IMG, 3))
x = layers.Rescaling(1 / 127.5, offset=-1)(inputs)   # device: (pixel / 127.5) - 1
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

def compile_model(lr):
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

early = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4,
                                         restore_best_weights=True)

# ---------- stage 1: train head ----------
compile_model(1e-3)
model.fit(train, validation_data=val_deg, epochs=EPOCHS_HEAD,
          class_weight=class_weight, callbacks=[early])

# ---------- stage 2: fine-tune top layers, BatchNorm frozen ----------
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
compile_model(1e-4)
model.fit(train, validation_data=val_deg, epochs=EPOCHS_FINE,
          class_weight=class_weight, callbacks=[early])

print("clean val acc:", model.evaluate(val_clean, verbose=0)[1])
print("degraded val acc:", model.evaluate(val_deg, verbose=0)[1])
model.save(os.path.join(OUT_DIR, "organic_paper_plastic.keras"))

# ---------- export int8 TFLite ----------
def rep_data():
    for x, _ in train_raw.map(esp32_degrade, AUTO).take(200):
        yield [tf.expand_dims(x, 0)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = rep_data
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_model = converter.convert()

tflite_path = os.path.join(OUT_DIR, "trash_int8.tflite")
with open(tflite_path, "wb") as f:
    f.write(tflite_model)
print("TFLite size (KB):", round(len(tflite_model) / 1024, 1))

# ---------- verify int8 model on degraded validation images ----------
interp = tf.lite.Interpreter(model_content=tflite_model)
interp.allocate_tensors()
inp = interp.get_input_details()[0]
outp = interp.get_output_details()[0]
in_scale, in_zp = inp["quantization"]
out_scale, out_zp = outp["quantization"]

correct, total = 0, 0
per_class = np.zeros((NUM_CLASSES, 2), dtype=int)   # [correct, total]
for x, y in val_raw.map(esp32_degrade, AUTO):
    q = np.clip(np.round(x.numpy() / in_scale) + in_zp, -128, 127).astype(np.int8)
    interp.set_tensor(inp["index"], q[None])
    interp.invoke()
    raw = interp.get_tensor(outp["index"])[0].astype(np.float32)
    pred = int(np.argmax(raw))
    y = int(y)
    per_class[y, 1] += 1
    if pred == y:
        per_class[y, 0] += 1
        correct += 1
    total += 1

print("int8 degraded val acc:", round(correct / total, 3))
for i, name in enumerate(class_names):
    c, t = per_class[i]
    print(f"  {name}: {c}/{t} = {c / max(t, 1):.3f}")

# ---------- metadata for firmware ----------
meta = {
    "classes": class_names,
    "input_shape": [1, IMG, IMG, 3],
    "input_scale": float(in_scale),
    "input_zero_point": int(in_zp),
    "output_scale": float(out_scale),
    "output_zero_point": int(out_zp),
    "preprocess": "RGB888 0-255 -> q = round(pixel / input_scale) + input_zero_point, clamp to int8",
}
with open(os.path.join(OUT_DIR, "model_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)
print(json.dumps(meta, indent=2))

Found 3791 files belonging to 3 classes.
Using 3033 files for training.
Found 3791 files belonging to 3 classes.
Using 758 files for validation.
classes: ['organic', 'paper', 'plastic']
train counts: {'organic': 777, 'paper': 1573, 'plastic': 683}
val images: 758
2019640/2019640 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/8
95/95 ━━━━━━━━━━━━━━━━━━━━ 29s 181ms/step - accuracy: 0.7319 - loss: 0.6141 - val_accuracy: 0.8325 - val_loss: 0.4032
Epoch 2/8
95/95 ━━━━━━━━━━━━━━━━━━━━ 7s 62ms/step - accuracy: 0.8470 - loss: 0.3490 - val_accuracy: 0.8839 - val_loss: 0.3150
Epoch 3/8
95/95 ━━━━━━━━━━━━━━━━━━━━ 7s 63ms/step - accuracy: 0.8757 - loss: 0.2976 - val_accuracy: 0.8905 - val_loss: 0.2737
Epoch 4/8
95/95 ━━━━━━━━━━━━━━━━━━━━ 7s 63ms/step - accuracy: 0.8813 - loss: 0.2836 - val_accuracy: 0.8813 - val_loss: 0.3231
Epoch 5/8
95/95 ━━━━━━━━━━━━━━━━━━━━ 7s 61ms/step - accuracy: 0.8823 - loss: 0.2723 - val_accuracy: 0.8773 - val_loss: 0.2908
Epoch 6/8
95/95 ━━━━━━━━━━━━━━━━━━━━ 10s 62ms/step - ac

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


TFLite size (KB): 630.0


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


int8 degraded val acc: 0.892
  organic: 196/208 = 0.942
  paper: 347/368 = 0.943
  plastic: 133/182 = 0.731
{
  "classes": [
    "organic",
    "paper",
    "plastic"
  ],
  "input_shape": [
    1,
    96,
    96,
    3
  ],
  "input_scale": 1.0,
  "input_zero_point": -128,
  "output_scale": 0.00390625,
  "output_zero_point": -128,
  "preprocess": "RGB888 0-255 -> q = round(pixel / input_scale) + input_zero_point, clamp to int8"
}


In [4]:
import json, os
import numpy as np
import tensorflow as tf

OUT_DIR = "/content/out"
MODEL_PATH = f"{OUT_DIR}/trash_int8.tflite"
META_PATH = f"{OUT_DIR}/model_meta.json"

with open(META_PATH) as f:
    meta = json.load(f)
CLASSES = meta["classes"]
IMG = meta["input_shape"][1]

interp = tf.lite.Interpreter(model_path=MODEL_PATH)
interp.allocate_tensors()
inp = interp.get_input_details()[0]
outp = interp.get_output_details()[0]

def predict(img_rgb_uint8):
    """img_rgb_uint8: HxWx3 uint8 array already sized to IMG x IMG."""
    x = img_rgb_uint8.astype(np.float32)
    s, zp = meta["input_scale"], meta["input_zero_point"]
    q = np.clip(np.round(x / s) + zp, -128, 127).astype(np.int8)
    interp.set_tensor(inp["index"], q[None])
    interp.invoke()
    raw = interp.get_tensor(outp["index"])[0].astype(np.float32)
    probs = (raw - meta["output_zero_point"]) * meta["output_scale"]
    return probs


In [14]:
from google.colab import files
from PIL import Image
for name in files.upload():
    probs = predict(np.array(Image.open(name).convert("RGB").resize((IMG, IMG))))
    scores = "  ".join(f"{c}={p:.3f}" for c, p in zip(CLASSES, probs))
    print(f"{name}: {CLASSES[int(np.argmax(probs))]}   ({scores})")

Saving frame_20261008_123804.jpg to frame_20261008_123804.jpg
frame_20261008_123804.jpg: plastic   (organic=0.000  paper=0.027  plastic=0.973)
